# ARL-ADKR Experiment Figures


In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

OUT_DIR = Path("figures/arl_adkr_experimentsum")
OUT_DIR.mkdir(parents=True, exist_ok=True)
N = np.array([32, 48, 64, 96, 128])
SERIES = ["ARL high", "Practical orig", "Practical high"]
COLORS = ["#d62728", "#1f77b4", "#2ca02c"]
MARKERS = ["D", "o", "^"]

mpl.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.family": "DejaVu Serif",
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.26, "grid.linewidth": 0.5,
    "lines.linewidth": 1.8, "lines.markersize": 5, "pdf.fonttype": 42,
})

def savefig(fig, name, rect=None):
    fig.tight_layout(rect=rect)
    for suffix in ("png", "pdf"):
        fig.savefig(OUT_DIR / f"{name}.{suffix}", bbox_inches="tight")

def plot_series(ax, dataset, series=SERIES):
    for label, color, marker in zip(series, COLORS, MARKERS):
        mean, std = dataset[label]
        legend_label = {
            "ARL high": "ARL",
            "Practical orig": "Practical ADKR orig",
            "Practical high": "Practical ADKR high",
        }[label]
        ax.errorbar(N, mean, yerr=std, label=legend_label, color=color, marker=marker,
                    capsize=2.5, elinewidth=0.8)

DATA_PATH = Path("experiment_data.md")
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / "data" / "experiment_data.md"
DATA_TEXT = DATA_PATH.read_text(encoding="utf-8")

def _table_after(marker, required_columns):
    lines = DATA_TEXT.splitlines()
    start = next(i for i, line in enumerate(lines) if marker in line)
    for i in range(start + 1, len(lines) - 1):
        if "|" not in lines[i] or "|" not in lines[i + 1]:
            continue
        header = [part.strip() for part in lines[i].strip().strip("|").split("|")]
        if not all(column in header for column in required_columns):
            continue
        rows = []
        for line in lines[i + 2:]:
            if "|" not in line:
                break
            values = [part.strip() for part in line.strip().strip("|").split("|")]
            if len(values) == len(header):
                rows.append(dict(zip(header, values)))
        return rows
    raise ValueError(f"table not found after {marker!r}")

def _numbers(value):
    return [float(item) for item in re.findall(r"-?\d+(?:\.\d+)?", value)]

def _dataset(marker, labels):
    rows = _table_after(marker, ["n", *labels])
    means, stds = {label: [] for label in labels}, {label: [] for label in labels}
    for row in rows:
        for label in labels:
            values = _numbers(row[label])
            means[label].append(values[0])
            stds[label].append(values[1] if len(values) > 1 else 0.0)
    return {label: (np.array(means[label]), np.array(stds[label])) for label in labels}

PARAMETER_LABELS = ["fo=fn", "K=fo+1", "L=no−fo", "Practical κ(orig/p2)",
                    "Practical κ(high)", "cprop (orig)", "cprop (high)", "cval", "qval"]
parameter_rows = _table_after("## 0.", ["n", *PARAMETER_LABELS])
PARAMETERS = {label: np.array([_numbers(row[label])[0] for row in parameter_rows]) for label in PARAMETER_LABELS}
E2E = _dataset("## 1.", ["ARL orig", "ARL high", "Practical orig", "Practical high"])
BW100 = _dataset("### 100 Mbps", ["ARL orig", "ARL high", "Practical orig", "Practical high"])
BW50 = _dataset("### 50 Mbps", ["ARL orig", "ARL high", "Practical orig", "Practical high"])
BANDWIDTH = np.array([1000, 500, 200, 100, 50])
BW_SCAN = _dataset("### n=128", ["ARL orig", "ARL high", "Practical orig", "Practical high"])
RECOVERY = _dataset("## 3.", ["ARL", "Practical orig", "Practical high"])
PER_NODE_TOTAL = _dataset("## 6.", ["ARL orig", "ARL high", "Practical orig", "Practical high"])
arc_rows = _table_after("## 7.", ["n", "ARC construction traffic (MB/node)", "Share of total traffic (%)"])
ARC_CONSTRUCTION_TRAFFIC = np.array([_numbers(row["ARC construction traffic (MB/node)"])[0] for row in arc_rows])
ARC_FRACTION = np.array([_numbers(row["Share of total traffic (%)"])[0] for row in arc_rows])

def split(dataset):
    return ({k: v[0] for k, v in dataset.items()}, {k: v[1] for k, v in dataset.items()})

## Fig. A: End-to-End Latency


In [ ]:
for dataset, name in [(E2E, "figA1_default_aws"), (BW100, "figA2_100mbps")]:
    fig, ax = plt.subplots(figsize=(5.4, 3.35))
    plot_series(ax, dataset)
    ax.set_xlabel("Committee size $n$")
    ax.set_xticks(N)
    ax.set_ylabel("Latency (s)")
    ax.legend(frameon=False)
    savefig(fig, name)
    plt.show()


## Fig. B: Communication


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.35))
recovery_styles = [("ARL", COLORS[0], MARKERS[0]),
                   ("Practical orig", COLORS[1], MARKERS[1]),
                   ("Practical high", COLORS[2], MARKERS[2])]
for label, color, marker in recovery_styles:
    mean, std = RECOVERY[label]
    legend_label = {
        "ARL": "ARL",
        "Practical orig": "Practical ADKR orig",
        "Practical high": "Practical ADKR high",
    }[label]
    ax.errorbar(N, mean, yerr=std, label=legend_label, color=color, marker=marker,
                capsize=2.5, elinewidth=0.8)
ax.set(xlabel="Committee size $n$", ylabel="Traffic (MB / node)")
ax.set_xticks(N)
ax.legend(frameon=False)
savefig(fig, "figB1_recovery_traffic")
plt.show()

fig, ax = plt.subplots(figsize=(5.4, 3.35))
plot_series(ax, PER_NODE_TOTAL)
ax.set(xlabel="Committee size $n$", ylabel="Traffic (MB / node)")
ax.set_xticks(N)
ax.legend(frameon=False)
savefig(fig, "figB2_total_traffic")
plt.show()


## Fig. C: Bandwidth Sweep at n=128


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.35))
x = np.arange(len(BANDWIDTH))
for label, color, marker in zip(SERIES, COLORS, MARKERS):
    mean, std = BW_SCAN[label]
    legend_label = {
        "ARL high": "ARL",
        "Practical orig": "Practical ADKR orig",
        "Practical high": "Practical ADKR high",
    }[label]
    ax.errorbar(x, mean, yerr=std, label=legend_label, color=color, marker=marker,
                capsize=2.5, elinewidth=0.8)
ax.set(xlabel="Bandwidth (Mbps / node)", ylabel="Latency (s)")
ax.set_xticks(x, BANDWIDTH)
ax.legend(frameon=False)
savefig(fig, "figC_bandwidth_sweep_n128")
plt.show()


## Fig. D: ARC Construction Communication


In [ ]:
ARC_BAR_COLOR = "#FF6B78"
x = np.arange(len(N))

fig, ax = plt.subplots(figsize=(5.4, 3.6))
ax2 = ax.twinx()
bars = ax.bar(x, ARC_CONSTRUCTION_TRAFFIC, color=ARC_BAR_COLOR, width=0.62,
              alpha=0.78, label="ARC construction traffic")
line, = ax2.plot(x, ARC_FRACTION, color=COLORS[1], marker="o",
                 label="Share of total traffic")
ax.set(xlabel="Committee size $n$", ylabel="ARC construction traffic (MB / node)")
ax2.set_ylabel("Share of total traffic (%)")
ax.set_xticks(x, N)
ax.set_ylim(0, 2.5)
ax2.set_ylim(0, 4.0)
ax2.grid(False)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_color("black")
ax2.spines["right"].set_linewidth(0.9)
ax2.tick_params(axis="y", colors="black")
ax2.yaxis.label.set_color("black")
ax.legend([bars, line], [bars.get_label(), line.get_label()], frameon=False,
          loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=1)
savefig(fig, "figD2_arc_traffic_dual_axis", rect=(0, 0, 1, 0.78))
plt.show()
